# Week 22: Multimodal Foundation Models & Vision-Language Architectures
## Notebook 01: LLaVA & Visual Instruction Tuning from Scratch

### 1. Conceptual Intuition & Real-World Use Case
How do Large Multimodal Models (LMMs like GPT-4V, LLaVA, Claude 3.5 Sonnet) understand images?
- Vision Encoders (CLIP ViT) output spatial patch embeddings: $\mathbb{R}^{P \times D_v}$.
- Language Models (LLaMA) accept textual word embeddings: $\mathbb{R}^{T \times D_l}$.
- The vector spaces are completely incompatible ($D_v \neq D_l$, and visual tokens mean nothing to LLM attention heads).

In 2023, Liu et al. (UW-Madison / Microsoft) created **LLaVA (Large Language and Vision Assistant)**:
- Connects a frozen Vision Encoder to a frozen LLM via a simple learnable **Linear Projection Layer (Projector)**!
- Trains the projector on image-text pairs to align visual tokens into the LLM's word embedding space.
- Fine-tunes the entire system on **Visual Instruction Tuning** conversations!

---

### 2. Mathematical Foundations: Token Alignment Projection
Given image $I$ and text prompt $X_{\text{text}}$:

1. **Vision Encoder (CLIP ViT-L/14):**
   $$Z_v = g(I) \in \mathbb{R}^{N_v \times D_v}$$
   Where $N_v = 576$ visual patch tokens (for $24 \times 24$ grid of $14 \times 14$ patches).

2. **Multimodal Projection Layer ($W$):**
   $$H_v = Z_v \cdot W + b \in \mathbb{R}^{N_v \times D_l}$$
   Where $W \in \mathbb{R}^{D_v \times D_l}$ projects visual dimension $D_v$ into language dimension $D_l$.

3. **Concatenation into LLM:**
   $$X_{\text{input}} = [H_v \parallel H_{\text{text}}] \in \mathbb{R}^{(N_v + N_t) \times D_l}$$
   The language model processes visual tokens as if they were foreign language words!

In [ ]:
import numpy as np

np.random.seed(42)

# Step 1: Simulate Vision Encoder Output (ViT)
num_patches = 4     # e.g. 4 visual image patches
d_vision = 6        # CLIP ViT feature dimension

visual_features = np.random.randn(num_patches, d_vision)
print(f"Visual Features Z_v shape: {visual_features.shape} (4 patches x {d_vision} dim)")

# Step 2: Implement LLaVA Linear Projector
d_language = 8      # LLM word embedding dimension (e.g. LLaMA hidden size)

W_proj = np.random.randn(d_vision, d_language) * 0.2
b_proj = np.zeros(d_language)

# Project visual tokens into Language Space: H_v = Z_v @ W + b
projected_visual_tokens = visual_features @ W_proj + b_proj
print(f"Projected Visual Tokens H_v shape: {projected_visual_tokens.shape} (4 tokens x {d_language} dim)")

# Step 3: Text Prompt Embeddings (e.g. "Describe this image:")
prompt_tokens = np.array([
    [0.1, -0.2, 0.4, 0.1, 0.5, -0.1, 0.2, 0.3],  # "Describe"
    [0.2, 0.1, -0.3, 0.4, 0.0, 0.2, -0.1, 0.1],  # "this"
    [-0.1, 0.3, 0.2, -0.1, 0.4, 0.1, 0.0, 0.2]   # "image"
])
print(f"Text Prompt Tokens shape: {prompt_tokens.shape} (3 tokens x {d_language} dim)")

# Step 4: Multimodal Sequence Concatenation [Visual_Tokens || Text_Tokens]
multimodal_input_sequence = np.concatenate([projected_visual_tokens, prompt_tokens], axis=0)

print(f"\n--- Multimodal Sequence Fed to LLM Attention ---")
print("Total Sequence Length:", multimodal_input_sequence.shape[0], "(4 Image tokens + 3 Text tokens)")
print("Feature Dimension:    ", multimodal_input_sequence.shape[1])
print("First Visual Token in Language Space:\n", np.round(multimodal_input_sequence[0], 3))

### 3. Step-by-Step Code Walkthrough
1. **Dimension Bridging:** The projector projected the 6D visual space into the 8D language embedding space.
2. **Unified Sequence:** Visual tokens and text tokens were concatenated along the sequence axis.
3. **Standard Attention:** The downstream LLM evaluates standard causal multi-head self-attention across the combined sequence without requiring specialized vision kernels.

---

### 4. Industry Applications & Trade-offs
- **Medical Chart & Document VQA:** Analyzing X-rays and structured financial invoices.
- **Trade-off:** High token footprint: a single high-resolution image consumes 576+ tokens, consuming substantial context window space.

---

### 5. Top Interview Questions & Answers
**Q1: What is the 2-stage training procedure of LLaVA?**
> **Answer:** **Stage 1 (Feature Alignment Pre-training):** Freeze both the Vision Encoder and the LLM; train *only* the projector matrix on CC3M image-caption pairs to map visual features into word space. **Stage 2 (Visual Instruction Tuning):** Keep the vision encoder frozen, unfreeze the projector and LLM, and train on multi-turn conversational question-answering data.

**Q2: What is the difference between LLaVA's linear projector and BLIP-2's Q-Former?**
> **Answer:** BLIP-2 uses a multi-layer Q-Former (a small Transformer with cross-attention) that uses learned query tokens to extract a fixed number of visual features (e.g. 32 tokens). LLaVA uses a simple 2-layer MLP projection that preserves all 576 spatial patch tokens, which is simpler and preserves fine-grained spatial localization.